# Week 5 · Day 14 — Capstone Build — Session 2: Refine, Test, Document

**Course:** IPAM USL 5-Week Short Course: Introduction to Artificial Intelligence *(Introductory tier)*
**Facilitator:** Solomon Wilson MBCS | Deputy HOD Transport Planning & Operations | IT & Audit Supervisor, SLPTA
**Mode:** Google Colab (zero-install)
**Mental model layer:** L14 — Capstone: Refine
**Running scenario:** Route **R12** (Wilberforce → CBD) — operator OP-104, 25-minute delay
**Module:** 3 · **Week:** 5 · **Tier:** Intro
**New concept:** (Capstone build session 2 — no new concept today)
**Deliverable wired in:** D3 — Capstone Notebook + Project Brief (40%) due end of session

## Learning objectives
By the end of today you will:
- **Test** your prototype, including awkward inputs.
- **Refine** at least one weakness you find.
- Write your one-page **project brief** (**Deliverable 3**).

## Why this matters

A demo that works once is not the same as a tool people can trust. Today you stress-test your capstone the way a real dispatcher would, fix what breaks, and write the brief that explains it to management.

## Environment setup

In [1]:
# Same stack as Session 1.
!pip install -q gradio google-genai
print("Environment ready.")

Environment ready.


In [2]:
# --- Standard SLPTA bootstrap (identical in every notebook) ----------------
import sys
from pathlib import Path
for candidate in [Path.cwd(), *Path.cwd().parents,
                  Path("/content/IPAM_USL_Intro_AI_5Week")]:
    if (candidate / "shared" / "slpta_bootstrap.py").exists():
        sys.path.insert(0, str(candidate / "shared"))
        break

from slpta_bootstrap import (MODEL, ensure_course_data, get_client,
                             load_route12_context, load_route_logs,
                             load_complaints, load_routes, load_operators)

ensure_course_data()
print("Model configured:", MODEL)
print(load_route12_context())

Model configured: gemini-2.0-flash
Route R12 (Wilberforce → CBD). The 07:45 service, operated by OP-104 on vehicle SLPTA-1142, departed 25 minutes late. Recorded cause: Heavy traffic on Wilkinson Road. About 40 passengers were affected and the dispatch desk received multiple complaints. (Synthetic SLPTA scenario — no real data.)


## Concept — test, then refine

Good engineering is a loop: **build → test (especially edge cases) → fix → repeat.** A model that handles only tidy inputs fails on the messy reality of a dispatch desk.

<p align="center"></p>

### Step 1 — Test with edge cases

<!-- cell-diagram:c09 -->
<p align="center"></p>

### Check your understanding (before running)
Capstone refine: run your model on hold-out rows and inspect mistakes.

**Predict:** Will errors cluster on high-delay outliers (600 min) or random rows?

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

# Rebuild the reference model (or import your own from Session 1).
df = load_route_logs().drop_duplicates(); df = df[df["delay_minutes"] < 180].copy()
df["weather"] = df["weather"].fillna(df["weather"].mode()[0])
df["passenger_count"] = df["passenger_count"].fillna(df["passenger_count"].median())
y = (df["delay_minutes"] > 15).astype(int)
X = pd.get_dummies(df[["route_id","peak_period","weather","cause_category",
                       "scheduled_hour","distance_km","passenger_count"]])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
model = DecisionTreeClassifier(max_depth=5, random_state=0).fit(Xtr, ytr)

print("Held-out accuracy:", f"{accuracy_score(yte, model.predict(Xte)):.1%}")
print("Recall (caught delays):", f"{recall_score(yte, model.predict(Xte)):.1%}")
print("Confusion matrix [[TN, FP],[FN, TP]]:\n", confusion_matrix(yte, model.predict(Xte)))

# TODO: write down which mistake (false positive vs false negative) hurts your
# users most, and whether your numbers are good enough to ship.

Held-out accuracy: 80.0%
Recall (caught delays): 51.1%
Confusion matrix [[TN, FP],[FN, TP]]:
 [[300  25]
 [ 68  71]]


### Edge Case Testing
Let's see how the model reacts to 'awkward' inputs, such as extremely high passenger counts or unusual weather categories that weren't in the training set (handled via the dummy columns).

<!-- cell-diagram:c11 -->
<p align="center"></p>

### Check your understanding (before running)
Capstone refine: run your model on hold-out rows and inspect mistakes.

**Predict:** Will errors cluster on high-delay outliers (600 min) or random rows?

In [5]:
import numpy as np

# Create an 'Edge Case' sample: High passengers during peak hour
# We initialize with a valid row and then set specific values to avoid dtype warnings
edge_case = Xte.iloc[0:1].copy()
for col in edge_case.columns:
    if edge_case[col].dtype == 'bool':
        edge_case[col] = False
    else:
        edge_case[col] = 0

edge_case['passenger_count'] = 150  # Overloaded bus
edge_case['peak_period'] = True
edge_case['scheduled_hour'] = 8

prediction = model.predict(edge_case)
probability = model.predict_proba(edge_case)

print(f"Edge Case Prediction (1=Delay): {prediction[0]}")
print(f"Confidence: {np.max(probability)*100:.1f}%")

# Reflection question for your brief:
# If a bus is 3x over capacity, does your model automatically assume a delay?
# If not, you might need more data on capacity vs delay."

Edge Case Prediction (1=Delay): 1
Confidence: 96.1%


### Step 2 — Peer test checklist
Swap notebooks with a partner. As the "dispatcher", try to break their tool, then fill this in.

| Test | Result (pass / fix needed) |
|------|----------------------------|
| Does it run end-to-end from a fresh runtime? | |
| Does it handle an empty or odd input gracefully? | |
| Is the output clear to a non-technical user? | |
| Does it state its limitations honestly? | |

## Deliverable 3 — Project Brief (graded, part of the 40% capstone code mark)

One page. Fill in below. *Submit with your notebook before Day 15.*

### My project brief
*Double-click to edit.*

**Project title:**

**1. Problem** (what SLPTA need, in 2-3 sentences):

**2. Data** (which `course_data/` file(s) and columns):

**3. Model / API used** (and why):

**4. Results** (your key numbers — accuracy, MAE, recall, or example outputs):

**5. Limitations** (where it is weak; when a human must check it):

**6. Real-world deployment path** (how SLPTA could actually use this):

### Check your understanding (before submitting D3)
Answer all three before you submit:
1. **What problem does your capstone solve for SLPTA?** (One sentence.)
2. **What is one thing your system cannot do?** (Honest limitation.)
3. **What would you improve with one more day of build time?**

> **D3 pass threshold:** Proficient (2/4) or above on **every** dimension of
> the rubric introduced in Day 13. Re-read it and self-assess before submitting.

> **If you remember one thing from today's build session:**
> A working prototype that does one thing well is more valuable than
> an ambitious system that does nothing reliably.

## Submission checklist
- [ ] Edge-case tests run and documented
- [ ] At least one refinement made and noted
- [ ] Peer-test checklist completed
- [ ] **Deliverable 3** project brief filled in
- [ ] Save a clean copy of the notebook (*File → Save a copy in Drive*)